# QuickShield Dynamic Premium Calculation using Risk Score

### 1.Installing XGBoost & setting up all other algos needed

In [1]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Importing necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

### 2. Data Importing & Analysis

In [4]:
# load the dataset
climate_df = pd.read_csv('Indian_Climate_Dataset_2024_2025.csv')
worker_df = pd.read_csv('quickshield_dataset_final.csv')

In [5]:
worker_df

,worker_id,date,earnings,hours,deliveries,rating,acceptance_rate,cancellations,weekly_variance,tenure_months
0,W001,01-01-2026,318.72,8.93,12,4.76,0.87,0,0.38,0.00
1,W001,02-01-2026,347.45,6.08,10,4.79,0.83,0,0.17,0.03
2,W001,03-01-2026,223.89,7.22,8,4.69,0.90,0,0.25,0.07
3,W001,04-01-2026,362.81,7.82,11,4.59,0.85,0,0.12,0.10
4,W001,05-01-2026,313.38,8.43,12,4.63,0.90,0,0.30,0.13
...,...,...,...,...,...,...,...,...,...,...
11995,W200,25-02-2026,520.68,9.53,19,4.54,0.90,0,0.37,1.83
11996,W200,26-02-2026,383.02,6.16,12,4.49,0.93,0,0.30,1.87
11997,W200,27-02-2026,507.88,7.83,16,4.55,0.85,0,0.35,1.90
11998,W200,28-02-2026,448.78,6.03,12,4.65,0.88,1,0.22,1.93


In [6]:
climate_df

,Date,City,State,Temperature_Max (°C),Temperature_Min (°C),Temperature_Avg (°C),Humidity (%),Rainfall (mm),Wind_Speed (km/h),AQI,AQI_Category,Pressure (hPa),Cloud_Cover (%)
0,2024-01-01,Mumbai,Maharashtra,32.5,18.0,25.2,77.6,0.0,3.3,259,Poor,1020.3,62.1
1,2024-01-01,Delhi,Delhi,25.4,10.7,18.1,84.1,0.0,9.0,130,Moderate,1008.4,46.0
2,2024-01-01,Bengaluru,Karnataka,37.2,30.8,34.0,49.0,3.7,6.6,54,Satisfactory,1008.0,61.3
3,2024-01-01,Chennai,Tamil Nadu,37.2,30.4,33.8,34.2,9.5,9.0,176,Moderate,993.4,70.0
4,2024-01-01,Kolkata,West Bengal,27.4,17.5,22.5,32.2,9.1,9.2,97,Satisfactory,1008.2,56.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7305,2025-12-31,Hyderabad,Telangana,41.8,32.8,37.3,65.2,4.2,19.4,95,Satisfactory,1007.0,48.3
7306,2025-12-31,Ahmedabad,Gujarat,37.7,23.5,30.6,76.7,12.7,9.1,173,Moderate,1000.5,49.6
7307,2025-12-31,Jaipur,Rajasthan,44.2,37.6,40.9,74.8,1.2,20.5,329,Very Poor,1006.4,80.0
7308,2025-12-31,Lucknow,Uttar Pradesh,43.7,35.3,39.5,50.3,0.0,7.3,80,Satisfactory,1011.6,45.9


In [7]:
climate_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7310 entries, 0 to 7309
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Date                  7310 non-null   object 
 1   City                  7310 non-null   object 
 2   State                 7310 non-null   object 
 3   Temperature_Max (°C)  7310 non-null   float64
 4   Temperature_Min (°C)  7310 non-null   float64
 5   Temperature_Avg (°C)  7310 non-null   float64
 6   Humidity (%)          7310 non-null   float64
 7   Rainfall (mm)         7310 non-null   float64
 8   Wind_Speed (km/h)     7310 non-null   float64
 9   AQI                   7310 non-null   int64  
 10  AQI_Category          7310 non-null   object 
 11  Pressure (hPa)        7310 non-null   float64
 12  Cloud_Cover (%)       7310 non-null   float64
dtypes: float64(8), int64(1), object(4)
memory usage: 742.6+ KB


In [8]:
worker_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   worker_id        12000 non-null  object 
 1   date             12000 non-null  object 
 2   earnings         12000 non-null  float64
 3   hours            12000 non-null  float64
 4   deliveries       12000 non-null  int64  
 5   rating           12000 non-null  float64
 6   acceptance_rate  12000 non-null  float64
 7   cancellations    12000 non-null  int64  
 8   weekly_variance  12000 non-null  float64
 9   tenure_months    12000 non-null  float64
dtypes: float64(6), int64(2), object(2)
memory usage: 937.6+ KB


In [9]:
worker_df.describe()

,earnings,hours,deliveries,rating,acceptance_rate,cancellations,weekly_variance,tenure_months
count,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000
mean,406.664093,7.985546,13.028917,4.188104,0.859940,0.140083,0.299364,0.983333
std,103.568322,1.151528,3.021232,0.378125,0.050935,0.375240,0.115158,0.577365
min,131.300000,6.000000,5.000000,3.280000,0.670000,0.000000,0.100000,0.000000
25%,330.885000,6.987500,11.000000,3.880000,0.830000,0.000000,0.200000,0.492500
50%,398.195000,7.970000,13.000000,4.200000,0.860000,0.000000,0.300000,0.985000
75%,475.362500,8.990000,15.000000,4.500000,0.890000,0.000000,0.400000,1.477500
max,856.490000,10.000000,23.000000,5.000000,1.000000,3.000000,0.500000,1.970000


In [10]:
climate_df.describe()

,Temperature_Max (°C),Temperature_Min (°C),Temperature_Avg (°C),Humidity (%),Rainfall (mm),Wind_Speed (km/h),AQI,Pressure (hPa),Cloud_Cover (%)
count,7310.000000,7310.000000,7310.000000,7310.000000,7310.000000,7310.000000,7310.000000,7310.000000,7310.000000
mean,34.952161,25.006402,29.980192,62.653516,8.231300,13.522763,193.759508,1007.358577,52.630055
std,5.781372,6.476264,5.966698,18.680171,17.758208,6.562729,89.182569,10.109978,27.327729
min,25.000000,10.100000,17.600000,30.000000,0.000000,2.000000,40.000000,990.000000,5.000000
25%,30.000000,19.900000,25.000000,46.400000,0.000000,7.900000,116.000000,998.700000,28.925000
50%,34.900000,25.000000,30.000000,62.700000,0.000000,13.500000,194.000000,1007.300000,52.700000
75%,40.000000,30.000000,35.000000,78.700000,6.300000,19.100000,270.000000,1016.200000,76.200000
max,45.000000,39.800000,42.300000,95.000000,79.900000,25.000000,349.000000,1025.000000,100.000000


In [11]:
climate_df.isnull().sum()

Date                    0
City                    0
State                   0
Temperature_Max (°C)    0
Temperature_Min (°C)    0
Temperature_Avg (°C)    0
Humidity (%)            0
Rainfall (mm)           0
Wind_Speed (km/h)       0
AQI                     0
AQI_Category            0
Pressure (hPa)          0
Cloud_Cover (%)         0
dtype: int64

In [12]:
worker_df.isnull().sum()

worker_id          0
date               0
earnings           0
hours              0
deliveries         0
rating             0
acceptance_rate    0
cancellations      0
weekly_variance    0
tenure_months      0
dtype: int64

### 3. Pre-processing Data

In [13]:
climate_df['Date'] = pd.to_datetime(climate_df['Date'])
climate_df['month'] = climate_df['Date'].dt.month
climate_df['day'] = climate_df['Date'].dt.day

In [14]:
daily_climate = climate_df.groupby(['month', 'day']).agg({
    'Temperature_Avg (°C)': 'mean',
    'Rainfall (mm)': 'mean',
    'AQI': 'mean'
}).reset_index()

In [15]:
# Preprocess Worker Data: Ensure date format matches
worker_df['date'] = pd.to_datetime(worker_df['date'], format='%d-%m-%Y')
worker_df['month'] = worker_df['date'].dt.month
worker_df['day'] = worker_df['date'].dt.day

In [16]:
# Merge datasets
df = pd.merge(worker_df, daily_climate, on=['month', 'day'], how='left')

### 4. Setting of Risk Score

In [17]:
def calculate_risk_score(row):
    # 1. Temperature Risk (Ideal 25°C. Risk adds up as it goes to 0°C or 50°C)
    temp_risk = min(25, abs(row['Temperature_Avg (°C)'] - 25) * 1.0)
    
    # 2. AQI Risk (0-500 scale, higher is worse)
    aqi_risk = min(25, (row['AQI'] / 400) * 25)
    
    # 3. Rainfall Risk (Heavy monsoon risk)
    rain_risk = min(25, (row['Rainfall (mm)'] / 20) * 25)
    
    # 4. Stability Risk (Inverse Variance: Higher variance = Lower Risk)
    # Assuming 0.5 is the max variance in your dataset
    stability_risk = max(0, (1 - (row['weekly_variance'] / 0.5)) * 25)
    
    return temp_risk + aqi_risk + rain_risk + stability_risk

In [18]:
# Apply the formula
df['Risk_Score'] = df.apply(calculate_risk_score, axis=1)

In [19]:
# Visualize the result
print(df[['Risk_Score']].describe())

         Risk_Score
count  12000.000000
mean      37.495115
std        7.994752
min       16.446250
25%       31.733125
50%       37.440000
75%       43.229375
max       60.508125


### 5. Training the model

In [20]:
# Prepare features (Drop non-numeric and IDs)
X = df[['earnings', 'hours', 'deliveries', 'rating', 'acceptance_rate', 
        'cancellations', 'weekly_variance', 'Temperature_Avg (°C)', 
        'Rainfall (mm)', 'AQI']]
y = df['Risk_Score']

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
# Train XGBoost Regressor
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100)
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [23]:
# Save the model for Phase 2 Backend
model.save_model("quickshield_risk_model.json")

### 6. Evaluation & Visualization